<a id="ga-overview"></a>

$$
\text{遗传算法 (Genetic Algorithm, GA): 模仿生物进化 "物竞天择, 适者生存" 的随机搜索算法} \\ \quad \\
\begin{array}{|c|c|}
\hline
\text{生物} & \text{优化} \\
\hline
\text{个体 (Individual) / 染色体 (Chromosome)} & \text{一个解的编码 (例如一串 0/1)} \\
\text{基因 (Gene)} & \text{编码中的一位} \\
\text{种群 (Population)} & \text{同时维护的一组解} \\
\text{适应度 (Fitness)} & \text{目标函数值 (越大越好)} \\
\text{选择 (Selection)} & \text{适应度高的个体更可能留下后代} \\
\text{交叉 (Crossover)} & \text{两个父代交换部分基因, 组合出新解} \\
\text{变异 (Mutation)} & \text{随机改变少量基因, 保持多样性, 防止早熟} \\
\hline
\end{array} \\ \quad \\
\text{流程: 随机初始化种群} \to \text{计算适应度} \to \text{选择} \to \text{交叉} \to \text{变异} \to \text{新一代种群} \to \cdots \text{ 重复 G 代} \\
\text{精英保留 (Elitism): 每一代直接把最好的几个个体复制到下一代, 保证最优解不会丢失}
$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from math import comb

np.set_printoptions(linewidth=np.inf)
np.set_printoptions(suppress=True)

def plot_nicely():
    for ax in plt.gcf().get_axes():
        ax.grid(True, which="both", alpha=0.3)
        if ax.get_legend_handles_labels()[0]:
            ax.legend(fontsize='small')
    plt.tight_layout()

<a id="encoding"></a>

$$
\text{二进制编码 (Binary Encoding): 把区间 [a, b] 上的实数编码成 L 位二进制串} \\
\text{L 位二进制可以表示整数 } 0, 1, \cdots, 2^L - 1 \text{, 线性映射到 [a, b]:} \\
x = a + \frac{b - a}{2^L - 1} \sum_{k=0}^{L-1} \text{bit}_k \, 2^{L-1-k} \\
\text{相邻两个编码之间的距离 (精度): } \Delta x = \frac{b - a}{2^L - 1} \\
\text{要求精度 } \varepsilon \text{: } \frac{b - a}{2^L - 1} \leq \varepsilon \implies 2^L \geq \frac{b - a}{\varepsilon} + 1 \implies L = \left\lceil \log_2\left(\frac{b - a}{\varepsilon} + 1\right) \right\rceil \\
\text{例: [-1, 2], 精度 } 10^{-6} \text{: } L = \lceil \log_2(3 \times 10^6 + 1) \rceil = 22
$$

In [ ]:
a_lo, b_hi, eps = -1.0, 2.0, 1e-6
L_bits = int(np.ceil(np.log2((b_hi - a_lo) / eps + 1)))
powers = 2 ** np.arange(L_bits - 1, -1, -1)

def decode(bits):
    return a_lo + (b_hi - a_lo) * (bits @ powers) / (2**L_bits - 1)

def encode(x):
    k = int(round((x - a_lo) / (b_hi - a_lo) * (2**L_bits - 1)))
    return np.array([(k >> (L_bits - 1 - i)) & 1 for i in range(L_bits)])

print(f'需要的位数 L = {L_bits}, 实际精度 = {(b_hi - a_lo) / (2**L_bits - 1):.3e}')
for x in [-1.0, 0.123456, 1.85, 2.0]:
    bits = encode(x)
    print(f'x = {x:<9}: 编码 = {"".join(map(str, bits))}, 解码 = {decode(bits):.7f}, 误差 = {abs(decode(bits) - x):.2e}')

<a id="selection"></a>

$$
\text{1. 轮盘赌选择 (Roulette Wheel Selection): 被选中的概率正比于适应度 (要求 } f_i > 0 \text{)} \\
p_i = \frac{f_i}{\sum_{j=1}^N f_j} \\
\text{选 N 次 (有放回), 个体 i 被选中的次数服从二项分布 } B(N, p_i) \text{, 期望副本数:} \\
E[\text{副本数}] = N p_i = \frac{f_i}{\frac{1}{N}\sum_j f_j} = \frac{f_i}{\bar{f}} \\
\text{适应度高于平均值的个体期望会有多于 1 个后代, 低于平均值的少于 1 个} \\
\text{缺点: 适应度差别很小时几乎等于随机选; 有一个特别大时它会迅速占满种群 (早熟收敛)} \\ \quad \\
\text{2. 锦标赛选择 (Tournament Selection): 随机抽 k 个, 选其中最好的} \\
\text{排名第 r (从差到好, r = 1, \cdots, N) 的个体胜出的概率: 抽到的 k 个都不比它好, 且至少抽到了它自己} \\
P(\text{rank r 胜出}) = \left(\frac{r}{N}\right)^k - \left(\frac{r - 1}{N}\right)^k \quad (\text{有放回抽样}) \\
\text{只依赖排名, 不依赖适应度的具体数值, 不要求正数, k 越大选择压力越大}
$$

数学基础: [二项分布](../math_basics/statistics.ipynb#binomial)

In [ ]:
rng = np.random.default_rng(0)
fitness_demo = np.array([1.0, 2.0, 3.0, 4.0, 10.0])
N_demo = len(fitness_demo)

def roulette(fitness, n_select, rng):
    cum = np.cumsum(fitness / fitness.sum())
    return np.searchsorted(cum, rng.random(n_select))    # 转盘: 落在哪一段就选哪个

def tournament(fitness, n_select, k, rng):
    contestants = rng.integers(0, len(fitness), size=(n_select, k))
    return contestants[np.arange(n_select), np.argmax(fitness[contestants], axis=1)]

trials = 20000
counts_r = np.zeros(N_demo)
counts_t = np.zeros(N_demo)
for _ in range(trials):
    counts_r += np.bincount(roulette(fitness_demo, N_demo, rng), minlength=N_demo)
    counts_t += np.bincount(tournament(fitness_demo, N_demo, 2, rng), minlength=N_demo)
print(f'适应度 = {fitness_demo}, 平均 = {fitness_demo.mean()}')
print(f'轮盘赌: 模拟平均副本数 = {np.round(counts_r / trials, 3)}, 理论 f_i / f_bar = {np.round(fitness_demo / fitness_demo.mean(), 3)}')
r = np.arange(1, N_demo + 1)   # fitness_demo 已经从小到大排好, 排名 = 位置
p_tour = (r / N_demo)**2 - ((r - 1) / N_demo)**2
print(f'锦标赛 k=2: 模拟平均副本数 = {np.round(counts_t / trials, 3)}, 理论 N * P(胜出) = {np.round(N_demo * p_tour, 3)}')
print('适应度 10 和 4 差了很多, 但锦标赛只看排名, 所以最好的个体在锦标赛里没有轮盘赌那么占优势')

<a id="crossover-mutation"></a>

$$
\text{3. 单点交叉 (Single-Point Crossover): 以概率 } p_c \text{ 对两个父代随机选一个切点, 交换切点之后的部分} \\
\text{父代 1: } 110 | 01011 \quad \text{父代 2: } 001 | 11100 \implies \text{子代 1: } 110 | 11100 \quad \text{子代 2: } 001 | 01011 \\ \quad \\
\text{4. 变异 (Mutation): 每一位以概率 } p_m \text{ 翻转 (0 变 1, 1 变 0)} \\
\text{每个个体平均翻转 } L p_m \text{ 位 (二项分布的期望), 通常取 } p_m \approx \frac{1}{L} \text{, 平均每个个体变一位} \\ \quad \\
\text{5. 模式定理 (Schema Theorem, Holland): 为什么 GA 能工作的直观解释} \\
\text{模式 (Schema) H: 用 * 表示 "任意" 的模板, 例如 } H = 1{*}{*}0{*} \text{ 代表所有第 1 位是 1, 第 4 位是 0 的串} \\
o(H) \text{: 阶 (Order), 确定位的个数 (上例 = 2)} \\
\delta(H) \text{: 定义长度 (Defining Length), 第一个和最后一个确定位的距离 (上例 = 4 - 1 = 3)} \\
m(H, t) \text{: 第 t 代中符合模式 H 的个体数} \\ \quad \\
\text{推导下一代的期望个数:} \\
\text{选择: 每个符合 H 的个体期望副本数是 } \frac{f_i}{\bar{f}} \text{, 加起来} \implies m(H, t) \frac{f(H)}{\bar{f}} \quad (f(H) \text{: 符合 H 的个体的平均适应度}) \\
\text{交叉: 切点在 L - 1 个位置中均匀选, 只有落在定义长度内部才可能破坏模式, 概率} \leq p_c \frac{\delta(H)}{L - 1} \\
\text{变异: o(H) 个确定位都不能变, 存活概率 } (1 - p_m)^{o(H)} \geq 1 - o(H) p_m \text{ (伯努利不等式)} \\
E[m(H, t + 1)] \geq m(H, t) \frac{f(H)}{\bar{f}} \left[1 - p_c \frac{\delta(H)}{L - 1} - o(H) p_m\right] \\
\text{结论: 短 (} \delta \text{ 小), 低阶 (o 小), 适应度高于平均的模式 (积木块 Building Block) 在种群中按指数增长}
$$

In [ ]:
def crossover(parents, pc, rng):
    children = parents.copy()
    L = parents.shape[1]
    for i in range(0, len(parents) - 1, 2):
        if rng.random() < pc:
            cut = rng.integers(1, L)                       # 切点在 1..L-1 之间
            children[i, cut:], children[i + 1, cut:] = parents[i + 1, cut:], parents[i, cut:]
    return children

def mutate(pop, pm, rng):
    flip = rng.random(pop.shape) < pm
    return np.where(flip, 1 - pop, pop)

# 验证模式定理: 固定一个种群, 重复做一代 "选择 + 交叉 + 变异", 统计模式 H 的个数
rng = np.random.default_rng(1)
L_s, N_s = 10, 40
pop = rng.integers(0, 2, size=(N_s, L_s))
fit = 1 + pop[:, 0] * 3 + (1 - pop[:, 3]) * 2 + rng.random(N_s)          # 让 "第 1 位是 1, 第 4 位是 0" 的个体适应度高
H_pos, H_val = np.array([0, 3]), np.array([1, 0])                         # H = 1**0******
o_H, delta_H = len(H_pos), H_pos.max() - H_pos.min()
matches = lambda P: np.all(P[:, H_pos] == H_val, axis=1)

pc, pm = 0.8, 0.02
m_t = matches(pop).sum()
f_H = fit[matches(pop)].mean()
bound = m_t * f_H / fit.mean() * (1 - pc * delta_H / (L_s - 1) - o_H * pm)
next_counts = []
for _ in range(5000):
    parents = pop[roulette(fit, N_s, rng)]
    child = mutate(crossover(parents, pc, rng), pm, rng)
    next_counts.append(matches(child).sum())
print(f'o(H) = {o_H}, delta(H) = {delta_H}, m(H, t) = {m_t}, f(H) / f_bar = {f_H / fit.mean():.4f}')
print(f'模拟的 E[m(H, t+1)] = {np.mean(next_counts):.4f} >= 模式定理下界 {bound:.4f} ? {np.mean(next_counts) >= bound}')
print(f'变异: 每个个体平均翻转 {mutate(np.zeros((20000, L_s), int), pm, rng).sum(axis=1).mean():.4f} 位, 理论 L * pm = {L_s * pm}')

<a id="ga-function"></a>

$$
\text{例 1: 函数最大值} \\
\max f(x) = x \sin(10 \pi x) + 2, \quad x \in [-1, 2] \\
\text{振荡很快, 有几十个局部最大值, 用上面 22 位的二进制编码} \\
\text{适应度直接用 f(x) (在区间上 f > 0, 可以用轮盘赌)}
$$

In [ ]:
def f_ga(x):
    return x * np.sin(10 * np.pi * x) + 2

def genetic_algorithm(fitness_fn, L, N=60, G=150, pc=0.8, pm=None, n_elite=2, seed=0, selection='roulette'):
    rng = np.random.default_rng(seed)
    pm = pm if pm is not None else 1 / L
    pop = rng.integers(0, 2, size=(N, L))
    best_hist, mean_hist = [], []
    for g in range(G):
        fit = fitness_fn(pop)
        order = np.argsort(-fit)
        best_hist.append(fit[order[0]]); mean_hist.append(fit.mean())
        elite = pop[order[:n_elite]].copy()
        if selection == 'roulette':
            idx = roulette(fit - fit.min() + 1e-9, N - n_elite, rng)    # 平移成正数
        else:
            idx = tournament(fit, N - n_elite, 3, rng)
        children = mutate(crossover(pop[idx], pc, rng), pm, rng)
        pop = np.vstack([elite, children])
    fit = fitness_fn(pop)
    best = pop[np.argmax(fit)]
    return best, fit.max(), np.array(best_hist), np.array(mean_hist)

grid = np.linspace(-1, 2, 3000001)
x_true = grid[np.argmax(f_ga(grid))]
print(f'网格搜索: x = {x_true:.6f}, f = {f_ga(x_true):.6f}')

best_bits, best_f, best_hist, mean_hist = genetic_algorithm(lambda P: f_ga(decode(P)), L_bits)
print(f'遗传算法: x = {decode(best_bits):.6f}, f = {best_f:.6f}, 和网格搜索差 {abs(best_f - f_ga(x_true)):.2e}')

hits = [abs(genetic_algorithm(lambda P: f_ga(decode(P)), L_bits, seed=s)[1] - f_ga(x_true)) < 1e-3 for s in range(10)]
print(f'10 个随机种子中找到全局最大值 (误差 < 1e-3) 的次数 = {sum(hits)}')

fig, axs = plt.subplots(1, 2, figsize=(8, 3))
xx = np.linspace(-1, 2, 3000)
axs[0].plot(xx, f_ga(xx), 'k', linewidth=0.8)
axs[0].plot(decode(best_bits), best_f, 'r*', markersize=12, label='GA best')
axs[0].set_xlabel('x')
axs[1].plot(best_hist, label='best fitness')
axs[1].plot(mean_hist, label='mean fitness')
axs[1].set_xlabel('generation')
plot_nicely()
plt.show()

<a id="ga-knapsack"></a>

$$
\text{例 2: 0-1 背包 (和动态规划 / 整数规划同一个例子, 最优价值 84)} \\
\text{编码很自然: 第 i 位 = 1 表示放第 i 个物品, 不需要解码} \\
\text{约束处理: 超重的解用罚函数降低适应度} \\
\text{fitness}(x) = \sum_i v_i x_i - \lambda \max\left(0, \sum_i w_i x_i - W\right) \\
\text{罚系数 } \lambda \text{ 要比最大的性价比 } \max_i \frac{v_i}{w_i} \text{ 大, 否则超重反而 "划算"} \\
\text{(多装 1 单位重量最多多得 } \max \frac{v_i}{w_i} \text{ 价值, 被罚 } \lambda \text{, 所以 } \lambda > \max \frac{v_i}{w_i} \text{ 时超重一定不划算)}
$$

对比: [动态规划解背包](dynamic_programming.ipynb#knapsack-dp) · [整数规划解背包](integer_programming.ipynb#knapsack) · [罚函数法](nonlinear_programming.ipynb#penalty)

In [ ]:
values_k = np.array([24, 13, 23, 15, 16, 11, 8, 26, 5, 19, 17, 9])
weights_k = np.array([12, 7, 11, 8, 9, 6, 5, 14, 3, 10, 9, 4])
capacity = 42

# 动态规划求精确最优 (见 dynamic_programming.ipynb)
dp = np.zeros(capacity + 1, dtype=int)
for w_i, v_i in zip(weights_k, values_k):
    for w in range(capacity, w_i - 1, -1):          # 一维数组倒序更新 = 二维表的压缩版本
        dp[w] = max(dp[w], dp[w - w_i] + v_i)
print(f'动态规划最优价值 = {dp[capacity]}')

lam = 2 * np.max(values_k / weights_k)
print(f'最大性价比 = {np.max(values_k / weights_k):.4f}, 罚系数 lambda = {lam:.4f}')

def knapsack_fitness(P):
    return P @ values_k - lam * np.maximum(0, P @ weights_k - capacity)

results = []
for s in range(10):
    best, best_fit, hist_k, _ = genetic_algorithm(knapsack_fitness, len(values_k), N=40, G=80, seed=s, selection='tournament')
    results.append((best_fit, best @ weights_k, np.flatnonzero(best).tolist()))
for fit_s, w_s, items in results[:3]:
    print(f'GA: 价值 = {fit_s:.0f}, 重量 = {w_s}, 选中 = {items}')
print(f'10 个随机种子中得到最优价值 {dp[capacity]} 的次数 = {sum(np.isclose(r[0], dp[capacity]) for r in results)}, 全部可行? {all(r[1] <= capacity for r in results)}')
print('GA 不保证最优, 也不知道自己是否已经最优 (没有像分支定界那样的上界), 适合精确方法太慢的大规模问题')